# E1 8단계: 레벨 2 + 보정 사진 311장 (백그라운드 실행용)
6단계(레벨 2, 미세조정 64장)에서 과적합 의심 → 미세조정 사진만 64 → 311장으로 늘린다.
- 양자화 눈금을 정하는 보정(calibration)은 `calibset_size=64`로 그대로 둔다. 311장 파일의 **앞 64장 = 6단계와 같은 사진**이라 눈금 조건이 같다.
- 미세조정은 311장 전부를 쓴다. 바꾸는 것은 이것 하나.

**실행 방법**: 오른쪽 설정(GPU T4 x2, Internet On, Input 두 개: `kaggle_upload`, `kaggle_upload2`) 확인 →
오른쪽 위 **Save Version → Save & Run All (Commit)**. 브라우저를 닫아도 계속 돈다(약 2시간).
끝나면 해당 버전의 Output에서 `summary.csv`, `run_checks.jsonl`, `emu_eval.jsonl`, `logs.tgz` 다운로드.

In [ ]:
import glob, os, re, json, time, subprocess, shutil, csv
os.environ['CUDA_VISIBLE_DEVICES'] = '0'          # GPU 1개만 (조건 고정)
def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, executable='/bin/bash', capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-3000:])
    if check and r.returncode: raise RuntimeError(f'실패(exit {r.returncode}): {cmd[:150]}')
    return r.returncode
INP  = os.path.dirname(glob.glob('/kaggle/input/**/rn50_parsed.har', recursive=True)[0])
INP2 = os.path.dirname(glob.glob('/kaggle/input/**/calib_rn50_highres311_s0_nhwc_u8.npy', recursive=True)[0])
WORK, RUNS, VENV = '/kaggle/working', '/tmp/runs', '/tmp/hv'; PY = f'{VENV}/bin/python'
for d in [f'{WORK}/logs', f'{WORK}/env', RUNS]: os.makedirs(d, exist_ok=True)
print(INP, INP2, sorted(os.listdir(f'{INP2}/alls')))
sh(f'cd {INP} && sed "s#  .*/#  #" SHA256SUMS.txt | sha256sum -c', check=True)
sh(f'cd {INP2} && sha256sum -c SHA256SUMS.txt', check=True)

In [ ]:
# 설치: 노트북과 같은 Python 3.10 + 같은 패키지 + DFC 5.4.0 (Kaggle uv는 --python으로 직접 지정해야 함)
shutil.copy(glob.glob(f'{INP}/hailo_dataflow_compiler-5.4.0*.whl')[0], '/tmp/hailo_dataflow_compiler-5.4.0-py3-none-linux_x86_64.whl')
sh('nvidia-smi | tee /kaggle/working/env/nvidia_smi.txt')
sh('apt-get -qq update > /dev/null && apt-get -qq install -y graphviz graphviz-dev libgraphviz-dev > /dev/null', check=True)
sh('pip -q install uv', check=True)
sh(f'uv venv {VENV} --python 3.10 -q', check=True)
sh(f"grep -v -e '^hailo-dataflow-compiler' -e ' @ file:' {INP}/laptop_hailo10_venv_freeze.txt > /tmp/req.txt")
sh(f'UV_HTTP_TIMEOUT=300 uv pip install --python {PY} -r /tmp/req.txt 2>&1 | tail -3; exit ${{PIPESTATUS[0]}}', check=True)
sh(f'uv pip install --python {PY} --no-deps /tmp/hailo_dataflow_compiler-5.4.0-py3-none-linux_x86_64.whl 2>&1 | tail -2; exit ${{PIPESTATUS[0]}}', check=True)
sh(f'{VENV}/bin/hailo --version; uv pip freeze --python {PY} > {WORK}/env/kaggle_freeze.txt')
sh(f'''{PY} -c "import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print('GPU', len(g)); assert g" 2>&1 | grep -E "^GPU|Error" | tee {WORK}/env/tf_gpu.txt; exit ${{PIPESTATUS[0]}}''', check=True)

In [ ]:
def run(cfg, calib, tag):
    d = f'{RUNS}/{tag}'; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    log = f'{WORK}/logs/{tag}_optimize.log'; t = time.time()
    rc = sh(f'cd {d} && {VENV}/bin/hailo optimize {INP}/rn50_parsed.har --hw-arch hailo10h '
            f'--calib-set-path {calib} --model-script {INP2}/alls/{cfg}.alls > {log} 2>&1')
    txt = open(log, errors='ignore').read()
    chk = {'tag': tag, 'alls': cfg, 'calib': os.path.basename(calib), 'optimize_exit': rc, 'opt_sec': round(time.time() - t),
           'lowered_to_lv0': 'Reducing optimization level' in txt,
           'algorithms': sorted(set(m.strip() for m in re.findall(r'Starting ([A-Z][A-Za-z\- ]+?)(?:\s*$|\s*\()', txt, re.M))),
           'finetune_entries': (re.findall(r'Using dataset with (\d+) entries for finetune', txt) or [None])[-1]}
    har = glob.glob(f'{d}/*_optimized.har')
    if har:
        t = time.time()
        sh(f'cd {d} && {PY} {INP}/emu_eval.py --har {har[0]} --ctx quantized --data pets_dev --n 990 '
           f'--tag kaggle_{tag} --cache {INP} --out {WORK}/emu_eval.jsonl > {WORK}/logs/{tag}_eval.log 2>&1')
        chk['eval_sec'] = round(time.time() - t)
        rows = [json.loads(l) for l in open(f'{WORK}/emu_eval.jsonl') if f'"kaggle_{tag}"' in l]
        if rows: chk.update({k: rows[-1][k] for k in ('top1', 'top5', 'cos_mean', 'cos_p1')})
    shutil.rmtree(d, ignore_errors=True)        # .har(약 800MB)는 지워서 디스크 절약
    with open(f'{WORK}/run_checks.jsonl', 'a') as f: f.write(json.dumps(chk, ensure_ascii=False) + '\n')
    print(json.dumps(chk, ensure_ascii=False)); return chk

In [ ]:
C311 = f'{INP2}/calib_rn50_highres311_s0_nhwc_u8.npy'
PLAN = ['off8L2L3_lv2', 'off8L4_lv2', 'c3eq_lv2', 't1_off8L4add16_lv2', 't3_c3eqadd16_lv2']   # 중요한 순서
for cfg in PLAN:
    try: run(cfg, C311, cfg + '_c311')
    except Exception as e: print('ERROR', cfg, e)

In [ ]:
# 요약 (6단계 64장 결과와 나란히)
PREV64 = {'off8L2L3_lv2': 27.68, 'off8L4_lv2': 19.60, 'c3eq_lv2': 7.98}
LV0 = {'off8L2L3_lv2': 62.93, 'off8L4_lv2': 7.07, 'c3eq_lv2': 4.75}
rows = [json.loads(l) for l in open(f'{WORK}/run_checks.jsonl')]
with open(f'{WORK}/summary.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['tag', 'lv0_top1', 'lv2_c64_top1', 'lv2_c311_top1', 'top5', 'cos_mean', 'finetune_entries', 'opt_sec'])
    for r in rows:
        w.writerow([r['tag'], LV0.get(r['alls']), PREV64.get(r['alls']), r.get('top1'), r.get('top5'), r.get('cos_mean'), r.get('finetune_entries'), r['opt_sec']])
print(open(f'{WORK}/summary.csv').read())
sh(f'cd {WORK} && tar czf logs.tgz logs env && ls -la')